# Merge Felten Language Modeling AIOE into Panel

In [4]:
import pandas as pd

# Load the panel
panel = pd.read_csv('bls_onet_panel.csv')
print(f'Panel rows: {len(panel)}')
print(f'Panel OCC_CODE example: {panel["OCC_CODE"].iloc[0]}')

Panel rows: 202876
Panel OCC_CODE example: 11-1021


In [5]:
# Load Felten Language Modeling AIOE
felten = pd.read_excel('Language_Modeling_AIOE_and_AIIE.xlsx', sheet_name='LM AIOE')
felten = felten.rename(columns={
    'SOC Code': 'OCC_CODE',
    'Language Modeling AIOE': 'LM_AIOE'
})
print(f'Felten occupations: {len(felten)}')
print(felten.head())

Felten occupations: 774
  OCC_CODE                     Occupation Title   LM_AIOE
0  11-1011                     Chief Executives  1.308912
1  11-1021      General and Operations Managers  0.677615
2  11-2011  Advertising and Promotions Managers  1.217224
3  11-2021                   Marketing Managers  1.203774
4  11-2022                       Sales Managers  1.293821


In [6]:
# Merge — both use the same SOC format (e.g. 11-1011)
merged = panel.merge(felten[['OCC_CODE', 'LM_AIOE']],
                     on='OCC_CODE',
                     how='left')

matched = merged['LM_AIOE'].notna().sum()
total = len(merged)
print(f'Rows with AIOE score: {matched} / {total} ({matched/total*100:.1f}%)')

Rows with AIOE score: 171765 / 202876 (84.7%)


In [7]:
# Check AIOE distribution
print('AIOE score summary:')
print(merged['LM_AIOE'].describe().round(3))

# Top 10 most exposed occupations
print('\nTop 10 highest AIOE occupations:')
top = merged[['OCC_CODE','OCC_TITLE','LM_AIOE']].drop_duplicates('OCC_CODE').nlargest(10,'LM_AIOE')
print(top.to_string(index=False))

# Bottom 10
print('\nTop 10 lowest AIOE occupations:')
bottom = merged[['OCC_CODE','OCC_TITLE','LM_AIOE']].drop_duplicates('OCC_CODE').nsmallest(10,'LM_AIOE')
print(bottom.to_string(index=False))

AIOE score summary:
count    171765.000
mean          0.100
std           1.000
min          -1.854
25%          -0.896
50%           0.201
75%           1.076
max           1.926
Name: LM_AIOE, dtype: float64

Top 10 highest AIOE occupations:
OCC_CODE                                                    OCC_TITLE  LM_AIOE
 41-9041                                                Telemarketers 1.925633
 25-1123      English Language and Literature Teachers, Postsecondary 1.856860
 25-1124      Foreign Language and Literature Teachers, Postsecondary 1.813892
 25-1125                              History Teachers, Postsecondary 1.813362
 25-1112                                  Law Teachers, Postsecondary 1.801888
 25-1126              Philosophy and Religion Teachers, Postsecondary 1.799875
 25-1067                            Sociology Teachers, Postsecondary 1.770457
 25-1065                    Political Science Teachers, Postsecondary 1.769565
 25-1111 Criminal Justice and Law Enforcement

In [8]:
# Save final panel
merged.to_csv('bls_onet_felten_panel.csv', index=False)
print(f'Saved to bls_onet_felten_panel.csv — {len(merged)} rows')

Saved to bls_onet_felten_panel.csv — 202876 rows
